# BDD100K 객체 탐지 모델 학습 및 개선

이번 과제는 BDD100K 도로 이미지에서 객체의 위치와 종류를 찾는 객체 탐지 실험이다.  
23강 강의자료의 FCOS 방식 모델을 기준으로 사용하고, BDD100K 데이터 특성에 맞게 개선 모델을 만들었다.

BDD100K에는 자동차처럼 큰 객체도 있지만, 신호등과 표지판처럼 작은 객체도 많다.  
그래서 개선 방향은 작은 객체를 더 잘 보게 만들고, 예측 box가 너무 많이 생기는 문제를 줄이는 쪽으로 잡았다.

## 1. 환경 설정

필요한 라이브러리를 불러오고 seed를 고정한다.  
seed를 고정하면 실행할 때마다 데이터 섞임이나 초기값이 크게 달라지는 것을 줄일 수 있다.

In [ ]:
import os
import json
import math
import random
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.ops import nms, box_iou, generalized_box_iou_loss
import torchvision.transforms.functional as TF
from torchvision.transforms import ColorJitter

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 61
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. 데이터 경로 설정

이번 과제에서 사용하는 이미지는 `100k/val` 폴더의 10000장이다.  
데이터셋 안에는 segmentation용 폴더도 같이 들어있지만, 이번 과제는 box를 찾는 객체 탐지이므로 segmentation 폴더는 사용하지 않는다.

In [ ]:
DATA_ROOT = Path("./data")

def find_image_dir(root: Path):
    candidates = [
        root / "bdd100k" / "bdd100k" / "images" / "100k" / "val",
        root / "bdd100k" / "images" / "100k" / "val",
        root / "images" / "100k" / "val",
        root / "100k" / "val",
    ]

    for p in candidates:
        if p.exists():
            n_img = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png")))
            if n_img >= 10000:
                return p

    for p in root.rglob("val"):
        if not p.is_dir():
            continue
        s = str(p).lower()
        if "seg" in s or "color_labels" in s:
            continue
        n_img = len(list(p.glob("*.jpg"))) + len(list(p.glob("*.png")))
        if n_img >= 10000:
            return p

    raise FileNotFoundError("10000장의 val 이미지가 있는 폴더를 찾지 못했다.")

def find_label_json(root: Path):
    files = list(root.rglob("bdd100k_labels_images_val*.json"))
    if len(files) == 0:
        raise FileNotFoundError("bdd100k_labels_images_val JSON 파일을 찾지 못했다.")
    return files[0]

IMAGE_DIR = find_image_dir(DATA_ROOT)
LABEL_JSON = find_label_json(DATA_ROOT)

print("현재 작업 폴더:", Path.cwd())
print("IMAGE_DIR:", IMAGE_DIR)
print("이미지 개수:", len(list(IMAGE_DIR.glob("*.jpg"))) + len(list(IMAGE_DIR.glob("*.png"))))
print("LABEL_JSON:", LABEL_JSON)

## 3. 클래스와 데이터 분할

과제에서 지정한 10개 클래스만 사용한다.  
이미지는 파일 이름 기준으로 정렬한 뒤 앞 7000장을 학습 데이터, 뒤 3000장을 시험 데이터로 나눈다.

In [ ]:
CLASSES = [
    "pedestrian", "rider", "car", "truck", "bus",
    "train", "motorcycle", "bicycle", "traffic light", "traffic sign"
]

class_to_idx = {name: i for i, name in enumerate(CLASSES)}
idx_to_class = {i: name for name, i in class_to_idx.items()}
NUM_CLASSES = len(CLASSES)

image_files = sorted([p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in [".jpg", ".jpeg", ".png"]])
assert len(image_files) >= 10000, f"이미지가 10000장보다 적다. 현재 {len(image_files)}장"

image_files = image_files[:10000]
train_files = image_files[:7000]
test_files = image_files[7000:10000]

len(train_files), len(test_files), train_files[0].name, test_files[0].name

## 4. 라벨 JSON 읽기

JSON 라벨에는 이미지 이름과 객체 정보가 들어있다.  
각 객체에서 필요한 값은 클래스 이름과 box 좌표이다.

In [ ]:
with open(LABEL_JSON, "r", encoding="utf-8") as f:
    raw_labels = json.load(f)

len(raw_labels), raw_labels[0].keys()

## 5. 라벨 이름 정리

BDD100K 원본 라벨명과 과제 클래스명이 일부 다르다.  
예를 들어 원본의 `person`은 과제 클래스의 `pedestrian`에 해당한다. 이 부분을 맞추지 않으면 해당 객체가 학습에서 빠진다.

In [ ]:
category_alias = {
    "person": "pedestrian",
    "motor": "motorcycle",
    "bike": "bicycle",
}

label_map = defaultdict(list)
skipped_counter = Counter()

for item in raw_labels:
    name = item.get("name")
    labels = item.get("labels", [])

    for obj in labels:
        category = obj.get("category")
        category = category_alias.get(category, category)
        box = obj.get("box2d")

        if category not in class_to_idx:
            skipped_counter[category] += 1
            continue
        if box is None:
            continue

        x1 = float(box["x1"])
        y1 = float(box["y1"])
        x2 = float(box["x2"])
        y2 = float(box["y2"])

        if x2 <= x1 or y2 <= y1:
            continue

        label_map[name].append({
            "class": class_to_idx[category],
            "box": [x1, y1, x2, y2]
        })

counter = Counter()
for objs in label_map.values():
    for obj in objs:
        counter[idx_to_class[obj["class"]]] += 1

label_count_df = pd.DataFrame({
    "class": CLASSES,
    "count": [counter[c] for c in CLASSES]
})
label_count_df

## 6. 데이터셋 클래스

모델 입력 크기는 512로 사용했다.  
크기를 키우면 학습 시간은 늘어나지만, 작은 신호등과 표지판이 너무 작게 뭉개지는 문제를 줄일 수 있다.

box 좌표는 모델 입력 크기에 맞춰 pixel 좌표로 변환한다.  
FCOS 방식에서는 격자점에서 box의 네 변까지의 거리를 예측하기 때문에 pixel 좌표가 더 편하다.

In [ ]:
IMG_SIZE = 512

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

class BDDDetectionDataset(Dataset):
    def __init__(self, files, label_map, img_size=512, train=False):
        self.files = list(files)
        self.label_map = label_map
        self.img_size = img_size
        self.train = train
        self.color_jitter = ColorJitter(
            brightness=0.18,
            contrast=0.18,
            saturation=0.15,
            hue=0.02
        )

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = self.files[idx]
        image = Image.open(path).convert("RGB")
        orig_w, orig_h = image.size

        boxes = []
        labels = []

        for obj in self.label_map.get(path.name, []):
            x1, y1, x2, y2 = obj["box"]

            x1 = max(0, min(x1, orig_w - 1))
            x2 = max(0, min(x2, orig_w - 1))
            y1 = max(0, min(y1, orig_h - 1))
            y2 = max(0, min(y2, orig_h - 1))

            if x2 <= x1 or y2 <= y1:
                continue

            sx = self.img_size / orig_w
            sy = self.img_size / orig_h
            boxes.append([x1 * sx, y1 * sy, x2 * sx, y2 * sy])
            labels.append(obj["class"])

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.long)

        if self.train:
            if random.random() < 0.5:
                image = TF.hflip(image)
                if len(boxes) > 0:
                    x1 = boxes[:, 0].clone()
                    x2 = boxes[:, 2].clone()
                    boxes[:, 0] = self.img_size - x2
                    boxes[:, 2] = self.img_size - x1

            image = self.color_jitter(image)

        image = image.resize((self.img_size, self.img_size))
        image = TF.to_tensor(image)
        image = (image - MEAN) / STD

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": path.name
        }
        return image, target

def detection_collate_fn(batch):
    images = torch.stack([b[0] for b in batch], dim=0)
    targets = [b[1] for b in batch]
    return images, targets

train_ds = BDDDetectionDataset(train_files, label_map, IMG_SIZE, train=True)
test_ds = BDDDetectionDataset(test_files, label_map, IMG_SIZE, train=False)

BATCH_SIZE = 8

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=detection_collate_fn,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=detection_collate_fn,
    pin_memory=True
)

len(train_ds), len(test_ds)

## 7. 라벨 시각화 확인

학습 전에 정답 box가 제대로 그려지는지 먼저 확인한다.  
여기서 box 위치가 이상하면 모델 학습 결과도 정상적으로 나올 수 없다.

In [ ]:
def denormalize_image(img_tensor):
    img = img_tensor.detach().cpu() * STD + MEAN
    img = img.clamp(0, 1)
    return img.permute(1, 2, 0).numpy()

COLORS = [
    (230, 50, 50), (50, 160, 60), (50, 90, 230), (220, 170, 30), (180, 60, 180),
    (80, 180, 180), (240, 120, 40), (120, 120, 220), (220, 80, 80), (80, 200, 120)
]

def draw_boxes_on_image(image_np, boxes, labels, scores=None, width=2):
    img = Image.fromarray((image_np * 255).astype(np.uint8))
    draw = ImageDraw.Draw(img)

    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = [float(v) for v in box]
        label = int(labels[i])
        color = COLORS[label % len(COLORS)]
        name = idx_to_class[label]
        text = name if scores is None else f"{name} {float(scores[i]):.2f}"

        draw.rectangle([x1, y1, x2, y2], outline=color, width=width)
        draw.text((x1, max(0, y1 - 12)), text, fill=color)

    return np.array(img)

sample_img, sample_target = train_ds[0]
vis = draw_boxes_on_image(
    denormalize_image(sample_img),
    sample_target["boxes"].numpy(),
    sample_target["labels"].numpy()
)

plt.figure(figsize=(8, 8))
plt.imshow(vis)
plt.axis("off")
plt.title("train sample with ground truth boxes")
plt.show()

## 8. FCOS 타깃 만들기

23강 모델은 FCOS와 비슷한 anchor-free 방식이다.  
anchor-free 방식은 미리 정한 anchor box를 맞추는 대신, 각 격자점에서 box의 왼쪽, 위쪽, 오른쪽, 아래쪽 변까지의 거리를 예측한다.

한 격자점이 여러 객체 안에 들어갈 수 있다. 이 경우 더 작은 객체를 우선 배정했다. 작은 객체는 놓치기 쉬우므로 우선권을 주는 것이 좋다.

In [ ]:
def make_grid(feat_h, feat_w, stride, device):
    ys = (torch.arange(feat_h, device=device) + 0.5) * stride
    xs = (torch.arange(feat_w, device=device) + 0.5) * stride
    gy, gx = torch.meshgrid(ys, xs, indexing="ij")
    return torch.stack([gx, gy], dim=-1).reshape(-1, 2)

def ltrb_to_xyxy(centers, ltrb):
    cx, cy = centers[:, 0], centers[:, 1]
    return torch.stack([
        cx - ltrb[:, 0],
        cy - ltrb[:, 1],
        cx + ltrb[:, 2],
        cy + ltrb[:, 3],
    ], dim=-1)

def build_fcos_targets(targets, feat_h, feat_w, stride, device, center_radius=1.5):
    B = len(targets)
    centers = make_grid(feat_h, feat_w, stride, device)
    P = centers.shape[0]

    cls_t = torch.zeros(B, P, dtype=torch.long, device=device)
    reg_t = torch.zeros(B, P, 4, dtype=torch.float32, device=device)
    ctr_t = torch.zeros(B, P, dtype=torch.float32, device=device)
    pos_t = torch.zeros(B, P, dtype=torch.bool, device=device)

    cx = centers[:, 0]
    cy = centers[:, 1]

    for b, target in enumerate(targets):
        boxes = target["boxes"].to(device)
        labels = target["labels"].to(device)

        if boxes.numel() == 0:
            continue

        l = cx[:, None] - boxes[None, :, 0]
        t = cy[:, None] - boxes[None, :, 1]
        r = boxes[None, :, 2] - cx[:, None]
        bot = boxes[None, :, 3] - cy[:, None]
        ltrb = torch.stack([l, t, r, bot], dim=-1)
        inside_box = ltrb.min(dim=-1).values > 0

        box_cx = (boxes[:, 0] + boxes[:, 2]) / 2
        box_cy = (boxes[:, 1] + boxes[:, 3]) / 2
        radius = center_radius * stride
        center_x1 = torch.maximum(box_cx - radius, boxes[:, 0])
        center_y1 = torch.maximum(box_cy - radius, boxes[:, 1])
        center_x2 = torch.minimum(box_cx + radius, boxes[:, 2])
        center_y2 = torch.minimum(box_cy + radius, boxes[:, 3])

        cl = cx[:, None] - center_x1[None, :]
        ct = cy[:, None] - center_y1[None, :]
        cr = center_x2[None, :] - cx[:, None]
        cb = center_y2[None, :] - cy[:, None]
        inside_center = torch.stack([cl, ct, cr, cb], dim=-1).min(dim=-1).values > 0

        candidate = inside_box & inside_center

        # center sampling 때문에 너무 작은 객체가 전부 빠지는 경우를 막는다.
        no_candidate = candidate.sum(dim=0) == 0
        if no_candidate.any():
            candidate[:, no_candidate] = inside_box[:, no_candidate]

        areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        areas = areas[None, :].repeat(P, 1)
        areas[~candidate] = 1e9

        min_area, gt_idx = areas.min(dim=1)
        pos = min_area < 1e9

        if pos.sum() == 0:
            continue

        selected = gt_idx[pos]
        cls_t[b, pos] = labels[selected]
        reg_t[b, pos] = ltrb[pos, selected]
        pos_t[b, pos] = True

        lr = reg_t[b, pos][:, [0, 2]]
        tb = reg_t[b, pos][:, [1, 3]]
        ctr_t[b, pos] = torch.sqrt(
            (lr.min(dim=1).values / lr.max(dim=1).values.clamp(min=1e-6)) *
            (tb.min(dim=1).values / tb.max(dim=1).values.clamp(min=1e-6))
        )

    return cls_t, reg_t, ctr_t, pos_t, centers

## 9. 강의자료 모델

강의자료 모델은 ResNet-18의 앞부분을 backbone으로 쓰고, class/regression/centerness 세 개의 head를 붙인 구조이다.  
class head는 객체 종류를 예측하고, regression head는 box의 네 방향 거리를 예측한다. centerness head는 격자점이 객체 중심에 가까운지 판단한다.

In [ ]:
class ConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1, groups=8):
        super().__init__()
        padding = kernel_size // 2
        g = min(groups, out_ch)
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=False),
            nn.GroupNorm(g, out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class LectureFCOS(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=False):
        super().__init__()
        weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        bb = models.resnet18(weights=weights)
        self.backbone = nn.Sequential(
            bb.conv1, bb.bn1, bb.relu, bb.maxpool,
            bb.layer1, bb.layer2, bb.layer3,
        )
        self.stride = 16
        self.neck = nn.Sequential(
            ConvGNAct(256, 128, 3, 1),
            ConvGNAct(128, 128, 3, 1),
        )
        self.cls_head = nn.Conv2d(128, num_classes, 3, padding=1)
        self.reg_head = nn.Conv2d(128, 4, 3, padding=1)
        self.ctr_head = nn.Conv2d(128, 1, 3, padding=1)
        nn.init.constant_(self.cls_head.bias, -math.log((1 - 0.01) / 0.01))

    def forward(self, x):
        f = self.neck(self.backbone(x))
        B, _, H, W = f.shape
        cls = self.cls_head(f).permute(0, 2, 3, 1).reshape(B, -1, NUM_CLASSES)
        reg = (F.softplus(self.reg_head(f)) * self.stride).permute(0, 2, 3, 1).reshape(B, -1, 4)
        ctr = self.ctr_head(f).permute(0, 2, 3, 1).reshape(B, -1)
        return {"cls": cls, "reg": reg, "ctr": ctr, "stride": self.stride, "feat_hw": (H, W)}

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

lecture_model = LectureFCOS(NUM_CLASSES, pretrained=False).to(device)
print("lecture params:", count_parameters(lecture_model))

## 10. 개선 모델

개선 모델은 backbone을 더 작게 만들고, 최종 특징맵의 stride를 8로 줄였다.  
stride가 16이면 512 이미지를 32x32 격자로 보고, stride가 8이면 64x64 격자로 본다.  
작은 객체 입장에서는 64x64처럼 더 촘촘한 격자가 유리하다.

전체 파라미터 수는 5M 이하로 유지했다.

In [ ]:
class DWConvGNAct(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=3, stride=1):
        super().__init__()
        padding = kernel_size // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, in_ch, kernel_size, stride, padding, groups=in_ch, bias=False),
            nn.GroupNorm(min(8, in_ch), in_ch),
            nn.SiLU(inplace=True),
            nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False),
            nn.GroupNorm(min(8, out_ch), out_ch),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class ImprovedFCOS(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.stride = 8
        self.backbone = nn.Sequential(
            ConvGNAct(3, 32, 3, 2),       # 512 -> 256
            DWConvGNAct(32, 64, 3, 2),    # 256 -> 128
            DWConvGNAct(64, 128, 3, 2),   # 128 -> 64
            DWConvGNAct(128, 160, 3, 1),  # 64 -> 64
            DWConvGNAct(160, 192, 3, 1),  # 64 -> 64
        )
        self.neck = nn.Sequential(
            ConvGNAct(192, 192, 3, 1),
            DWConvGNAct(192, 192, 3, 1),
            ConvGNAct(192, 160, 3, 1),
        )
        self.cls_head = nn.Sequential(
            ConvGNAct(160, 160, 3, 1),
            nn.Conv2d(160, num_classes, 3, padding=1),
        )
        self.reg_head = nn.Sequential(
            ConvGNAct(160, 160, 3, 1),
            nn.Conv2d(160, 4, 3, padding=1),
        )
        self.ctr_head = nn.Sequential(
            ConvGNAct(160, 160, 3, 1),
            nn.Conv2d(160, 1, 3, padding=1),
        )
        nn.init.constant_(self.cls_head[-1].bias, -math.log((1 - 0.01) / 0.01))

    def forward(self, x):
        f = self.neck(self.backbone(x))
        B, _, H, W = f.shape
        cls = self.cls_head(f).permute(0, 2, 3, 1).reshape(B, -1, NUM_CLASSES)
        reg = (F.softplus(self.reg_head(f)) * self.stride).permute(0, 2, 3, 1).reshape(B, -1, 4)
        ctr = self.ctr_head(f).permute(0, 2, 3, 1).reshape(B, -1)
        return {"cls": cls, "reg": reg, "ctr": ctr, "stride": self.stride, "feat_hw": (H, W)}

improved_model = ImprovedFCOS(NUM_CLASSES).to(device)

# ResNet-18 layer3까지의 backbone parameter 수와 비교한다.
_ref = models.resnet18(weights=None)
resnet18_layer3_backbone = nn.Sequential(
    _ref.conv1, _ref.bn1, _ref.relu, _ref.maxpool,
    _ref.layer1, _ref.layer2, _ref.layer3,
)

print("ResNet18 layer3 backbone params:", count_parameters(resnet18_layer3_backbone))
print("improved backbone params:", count_parameters(improved_model.backbone))
print("improved total params:", count_parameters(improved_model))
print("5M 이하 여부:", count_parameters(improved_model) <= 5_000_000)
print("backbone 제약 만족:", count_parameters(improved_model.backbone) < count_parameters(resnet18_layer3_backbone))

## 11. 손실함수

손실함수는 세 부분으로 나누었다.

1. class loss: 각 격자점이 어떤 객체를 나타내는지 학습한다. 배경 칸이 훨씬 많기 때문에 focal loss를 사용했다.
2. box loss: 예측 box와 정답 box가 얼마나 잘 겹치는지 학습한다. GIoU loss를 사용했다.
3. centerness loss: 객체 중심에 가까운 격자점의 점수를 높이도록 학습한다.

이 조합은 예측 box가 너무 많이 생기는 문제를 줄이는 데 도움이 된다.

In [ ]:
def make_class_weights(files, label_map):
    counter = Counter()
    for p in files:
        for obj in label_map.get(p.name, []):
            counter[obj["class"]] += 1

    counts = torch.tensor([counter[i] for i in range(NUM_CLASSES)], dtype=torch.float32)
    weights = 1.0 / torch.sqrt(counts.clamp(min=1.0))
    weights = weights / weights.mean()
    weights = weights.clamp(0.35, 3.0)
    weights = weights / weights.mean()
    return counts, weights

class_counts, class_weights = make_class_weights(train_files, label_map)
class_weights = class_weights.to(device)

pd.DataFrame({
    "class": CLASSES,
    "count": class_counts.numpy().astype(int),
    "weight": class_weights.detach().cpu().numpy(),
})

In [ ]:
def sigmoid_focal_loss(logits, cls_target, pos_mask, class_weights=None, alpha=0.25, gamma=2.0):
    onehot = torch.zeros_like(logits)
    b_idx, p_idx = torch.where(pos_mask)

    if len(b_idx) > 0:
        onehot[b_idx, p_idx, cls_target[pos_mask]] = 1.0

    prob = torch.sigmoid(logits)
    ce = F.binary_cross_entropy_with_logits(logits, onehot, reduction="none")
    pt = prob * onehot + (1.0 - prob) * (1.0 - onehot)
    weight = alpha * onehot + (1.0 - alpha) * (1.0 - onehot)

    if class_weights is not None:
        cw = class_weights.view(1, 1, -1)
        weight = torch.where(onehot > 0, weight * cw, weight)

    return (weight * (1.0 - pt).pow(gamma) * ce).sum()

def compute_loss(outputs, targets, class_weights=None):
    cls = outputs["cls"]
    reg = outputs["reg"]
    ctr = outputs["ctr"]
    stride = outputs["stride"]
    feat_h, feat_w = outputs["feat_hw"]

    cls_t, reg_t, ctr_t, pos, centers = build_fcos_targets(
        targets, feat_h, feat_w, stride, cls.device
    )

    n_pos = pos.sum().clamp(min=1)

    loss_cls = sigmoid_focal_loss(cls, cls_t, pos, class_weights=class_weights) / n_pos

    if pos.any():
        b_idx, p_idx = torch.where(pos)
        pred_box = ltrb_to_xyxy(centers[p_idx], reg[b_idx, p_idx])
        gt_box = ltrb_to_xyxy(centers[p_idx], reg_t[b_idx, p_idx])

        pred_box[:, [0, 2]] = pred_box[:, [0, 2]].clamp(0, IMG_SIZE)
        pred_box[:, [1, 3]] = pred_box[:, [1, 3]].clamp(0, IMG_SIZE)
        gt_box[:, [0, 2]] = gt_box[:, [0, 2]].clamp(0, IMG_SIZE)
        gt_box[:, [1, 3]] = gt_box[:, [1, 3]].clamp(0, IMG_SIZE)

        loss_giou = generalized_box_iou_loss(pred_box, gt_box, reduction="sum") / n_pos
        loss_l1 = F.l1_loss(reg[b_idx, p_idx], reg_t[b_idx, p_idx], reduction="sum") / n_pos / IMG_SIZE
        loss_ctr = F.binary_cross_entropy_with_logits(ctr[pos], ctr_t[pos], reduction="sum") / n_pos
    else:
        loss_giou = cls.sum() * 0.0
        loss_l1 = cls.sum() * 0.0
        loss_ctr = cls.sum() * 0.0

    total = loss_cls + loss_giou + 0.25 * loss_l1 + loss_ctr
    logs = {
        "total": float(total.detach().cpu()),
        "cls": float(loss_cls.detach().cpu()),
        "giou": float(loss_giou.detach().cpu()),
        "l1": float(loss_l1.detach().cpu()),
        "ctr": float(loss_ctr.detach().cpu()),
        "pos": int(pos.sum().detach().cpu()),
    }
    return total, logs

## 12. 학습 함수

optimizer는 AdamW를 사용했다.  
학습률은 OneCycleLR로 조절했다. 처음에는 학습률을 올려 빠르게 탐색하고, 뒤로 갈수록 낮춰서 안정적으로 수렴하게 만든다.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler=None, class_weights=None, scaler=None):
    model.train()
    logs_sum = defaultdict(float)
    total_loss = 0.0

    pbar = tqdm(loader, desc="train", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        amp_enabled = (device.type == "cuda")
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            outputs = model(images)
            loss, logs = compute_loss(outputs, targets, class_weights=class_weights)

        if scaler is not None and amp_enabled:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total_loss += float(loss.detach().cpu())
        for k, v in logs.items():
            logs_sum[k] += v

        pbar.set_postfix(loss=f"{float(loss.detach().cpu()):.3f}", pos=logs["pos"])

    n = len(loader)
    avg = {k: v / n for k, v in logs_sum.items()}
    return total_loss / n, avg

def fit_model(model, loader, epochs=10, lr=1e-3, weight_decay=1e-4, class_weights=None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=epochs * len(loader),
        pct_start=0.20,
        div_factor=10.0,
        final_div_factor=100.0
    )
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    history = []
    for epoch in range(1, epochs + 1):
        loss, logs = train_one_epoch(
            model, loader, optimizer, scheduler,
            class_weights=class_weights,
            scaler=scaler
        )
        row = {"epoch": epoch, "loss": loss, **logs}
        history.append(row)
        print(
            f"[{epoch:02d}/{epochs}] loss={loss:.4f} "
            f"cls={logs['cls']:.3f} giou={logs['giou']:.3f} "
            f"ctr={logs['ctr']:.3f} pos={logs['pos']:.1f}"
        )

    return pd.DataFrame(history)
def fit_model_early_stop(
    model,
    loader,
    epochs=30,
    lr=2e-3,
    weight_decay=1e-4,
    class_weights=None,
    patience=4,
    min_delta=0.005
):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=lr,
        total_steps=epochs * len(loader),
        pct_start=0.20,
        div_factor=10.0,
        final_div_factor=100.0
    )

    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

    history = []

    best_loss = float("inf")
    best_state = None
    bad_count = 0

    for epoch in range(1, epochs + 1):
        loss, logs = train_one_epoch(
            model,
            loader,
            optimizer,
            scheduler,
            class_weights=class_weights,
            scaler=scaler
        )

        row = {"epoch": epoch, "loss": loss, **logs}
        history.append(row)

        print(
            f"[{epoch:02d}/{epochs}] loss={loss:.4f} "
            f"cls={logs['cls']:.3f} giou={logs['giou']:.3f} "
            f"ctr={logs['ctr']:.3f} pos={logs['pos']:.1f}"
        )

        if best_loss - loss > min_delta:
            best_loss = loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_count = 0
        else:
            bad_count += 1
            print(f"early stopping count: {bad_count}/{patience}")

        if bad_count >= patience:
            print(f"Early stopping at epoch {epoch}. Best loss: {best_loss:.4f}")
            break


    if best_state is not None:
        model.load_state_dict({
            k: v.to(device)
            for k, v in best_state.items()
        })

    return pd.DataFrame(history)

## 13. 모델 학습

강의자료 모델은 비교 기준으로 사용한다.  
개선 모델보다 짧게 학습해도 되지만, 같은 데이터에서 실제로 어느 정도 성능이 나오는지 확인하기 위해 몇 epoch은 직접 학습한다.

In [ ]:
EPOCHS_LECTURE = 8

lecture_model = LectureFCOS(NUM_CLASSES, pretrained=False).to(device)
lecture_history = fit_model(
    lecture_model,
    train_loader,
    epochs=EPOCHS_LECTURE,
    lr=1e-3,
    weight_decay=1e-4,
    class_weights=class_weights
)
lecture_history

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(lecture_history["epoch"], lecture_history["loss"], marker="o")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Lecture model training loss")
plt.grid(True)
plt.show()

In [ ]:
@torch.no_grad()
def detect_for_mid_check(model, images, score_th=0.10, iou_th=0.5, pre_nms_topk=2000, max_det=40):
    model.eval()
    images = images.to(device, non_blocking=True)
    outputs = model(images)

    cls = outputs["cls"]
    reg = outputs["reg"]
    ctr = outputs["ctr"]
    stride = outputs["stride"]
    feat_h, feat_w = outputs["feat_hw"]

    centers = make_grid(feat_h, feat_w, stride, cls.device)
    scores_all = torch.sqrt(torch.sigmoid(cls) * torch.sigmoid(ctr)[..., None])

    results = []

    for i in range(images.shape[0]):
        scores_i = scores_all[i]
        keep_pos, keep_cls = torch.where(scores_i > score_th)

        if keep_pos.numel() == 0:
            results.append({
                "boxes": torch.empty((0, 4)),
                "scores": torch.empty((0,)),
                "labels": torch.empty((0,), dtype=torch.long),
            })
            continue

        scores = scores_i[keep_pos, keep_cls]

        if scores.numel() > pre_nms_topk:
            top_idx = torch.argsort(scores, descending=True)[:pre_nms_topk]
            keep_pos = keep_pos[top_idx]
            keep_cls = keep_cls[top_idx]
            scores = scores[top_idx]

        boxes = ltrb_to_xyxy(centers[keep_pos], reg[i, keep_pos])
        boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, IMG_SIZE)
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, IMG_SIZE)

        labels = keep_cls.long()

        offset = labels.float()[:, None] * (IMG_SIZE + 1)
        keep = nms(boxes + offset, scores, iou_th)
        keep = keep[:max_det]

        results.append({
            "boxes": boxes[keep].detach().cpu(),
            "scores": scores[keep].detach().cpu(),
            "labels": labels[keep].detach().cpu(),
        })

    return results


@torch.no_grad()
def visualize_lecture_before_improvement(indices=None, score_th=0.10, iou_th=0.5, max_det=40):
    lecture_model.eval()

    if indices is None:
        random.seed(SEED)
        indices = random.sample(range(len(test_ds)), 5)

    for idx in indices:
        img, target = test_ds[idx]
        img_np = denormalize_image(img)

        pred = detect_for_mid_check(
            lecture_model,
            img.unsqueeze(0),
            score_th=score_th,
            iou_th=iou_th,
            max_det=max_det
        )[0]

        gt_img = draw_boxes_on_image(
            img_np,
            target["boxes"].numpy(),
            target["labels"].numpy(),
            scores=None,
            width=2
        )

        pred_img = draw_boxes_on_image(
            img_np,
            pred["boxes"].numpy(),
            pred["labels"].numpy(),
            pred["scores"].numpy(),
            width=2
        )

        fig, axes = plt.subplots(1, 2, figsize=(12, 6))

        axes[0].imshow(gt_img)
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(pred_img)
        axes[1].set_title(f"Lecture Model Prediction / th={score_th}")
        axes[1].axis("off")

        plt.suptitle(target["image_id"])
        plt.tight_layout()
        plt.show()


random.seed(SEED)
before_improvement_indices = random.sample(range(len(test_ds)), 5)

visualize_lecture_before_improvement(
    indices=before_improvement_indices,
    score_th=0.10,
    iou_th=0.5,
    max_det=20
)

## 14. 개선 모델 학습

개선 모델은 더 촘촘한 특징맵을 사용하므로 작은 객체에 더 유리하다.  
대신 계산량이 늘어나기 때문에 batch size는 작게 두고, epoch을 더 길게 잡았다.

In [ ]:
EPOCHS_IMPROVED = 20

improved_model = ImprovedFCOS(NUM_CLASSES).to(device)

improved_history = fit_model_early_stop(
    improved_model,
    train_loader,
    epochs=EPOCHS_IMPROVED,
    lr=1.5e-3,
    weight_decay=1e-4,
    class_weights=class_weights,
    patience=5,
    min_delta=0.002
)

improved_history

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(improved_history["epoch"], improved_history["loss"], marker="o")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Improved model training loss")
plt.grid(True)
plt.show()

## 15. 예측 box 디코딩

모델 출력은 아직 최종 box가 아니다.  
각 격자점의 class 점수와 centerness 점수를 합쳐 confidence를 만들고, 겹치는 box는 NMS로 정리한다.

threshold가 너무 낮으면 box가 너무 많이 나오고, 너무 높으면 맞는 box도 사라진다. 그래서 여러 threshold를 비교한다.

In [ ]:
def _empty_pred():
    return {
        "boxes": torch.empty((0, 4)),
        "scores": torch.empty((0,)),
        "labels": torch.empty((0,), dtype=torch.long),
    }


def suppress_cross_class_duplicates(boxes, scores, labels, group_classes, iou_th=0.70):
    if boxes.numel() == 0:
        return torch.arange(0, device=labels.device)

    group = torch.tensor(group_classes, device=labels.device)
    group_mask = (labels[:, None] == group[None, :]).any(dim=1)
    group_idx = torch.where(group_mask)[0]

    keep_mask = torch.ones(labels.shape[0], dtype=torch.bool, device=labels.device)

    if group_idx.numel() == 0:
        return torch.where(keep_mask)[0]

    order = group_idx[torch.argsort(scores[group_idx], descending=True)]

    for pos, idx in enumerate(order):
        if not keep_mask[idx]:
            continue

        rest = order[pos + 1:]
        rest = rest[keep_mask[rest]]

        if rest.numel() == 0:
            continue

        ious = box_iou(boxes[idx].unsqueeze(0), boxes[rest]).squeeze(0)
        duplicated = rest[ious > iou_th]
        keep_mask[duplicated] = False

    return torch.where(keep_mask)[0]


def limit_predictions_per_class(boxes, scores, labels, limits):
    if scores.numel() == 0:
        return boxes, scores, labels

    keep = torch.ones(labels.shape[0], dtype=torch.bool, device=labels.device)

    for cls_id, limit in limits.items():
        idx = torch.where(labels == cls_id)[0]
        if idx.numel() > limit:
            order = idx[torch.argsort(scores[idx], descending=True)]
            keep[order[limit:]] = False

    return boxes[keep], scores[keep], labels[keep]


def apply_detection_rules(boxes, scores, labels, img_size=IMG_SIZE):
    if scores.numel() == 0:
        return boxes, scores, labels

    boxes = boxes.clone()
    labels = labels.clone()

    pedestrian = class_to_idx["pedestrian"]
    rider = class_to_idx["rider"]
    car = class_to_idx["car"]
    truck = class_to_idx["truck"]
    bus = class_to_idx["bus"]
    motorcycle = class_to_idx["motorcycle"]
    bicycle = class_to_idx["bicycle"]
    light = class_to_idx["traffic light"]
    sign = class_to_idx["traffic sign"]

    bw = boxes[:, 2] - boxes[:, 0]
    bh = boxes[:, 3] - boxes[:, 1]
    area = bw * bh
    area_ratio = area / float(img_size * img_size)
    aspect = bw / bh.clamp(min=1.0)
    y2 = boxes[:, 3]

    min_score = torch.full_like(scores, 0.050)

    # 작은 객체는 너무 세게 자르면 바로 미검출이 늘어서 약하게 둔다.
    min_score[labels == light] = 0.045
    min_score[labels == sign] = 0.050

    # truck/bus는 허공 오검출이 많아서 car보다 조금 더 보수적으로 둔다.
    min_score[labels == truck] = 0.095
    min_score[labels == bus] = 0.100

    valid = scores >= min_score

    # 전체 공통 비정상 box 제거
    valid &= (bw > 2)
    valid &= (bh > 2)
    valid &= (area > 4)
    valid &= (area < img_size * img_size * 0.78)
    valid &= (aspect > 0.05)
    valid &= (aspect < 16.0)

    # traffic light는 작고 세로로 긴 경우가 많다.
    light_rule = (
        (area >= 6) &
        (area_ratio <= 0.035) &
        (aspect >= 0.10) &
        (aspect <= 4.0)
    )

    # traffic sign은 light보다 조금 더 넓은 박스까지 허용한다.
    sign_rule = (
        (area >= 6) &
        (area_ratio <= 0.060) &
        (aspect >= 0.12) &
        (aspect <= 6.0)
    )

    # truck/bus가 이미지 위쪽 허공에 뜨는 경우를 줄인다.
    truck_rule = (
        (area >= 80) &
        (area_ratio >= 0.0015) &
        (area_ratio <= 0.55) &
        (bh >= 7) &
        (aspect >= 0.30) &
        (aspect <= 9.0) &
        (y2 >= img_size * 0.23)
    )

    bus_rule = (
        (area >= 90) &
        (area_ratio >= 0.0018) &
        (area_ratio <= 0.60) &
        (bh >= 7) &
        (aspect >= 0.35) &
        (aspect <= 10.0) &
        (y2 >= img_size * 0.23)
    )

    # 작은 truck/bus 후보는 특히 car와 헷갈리기 쉬우므로 점수가 더 높을 때만 둔다.
    small_truck_or_bus = ((labels == truck) | (labels == bus)) & (area_ratio < 0.018)
    valid &= torch.where(small_truck_or_bus, scores >= 0.130, torch.ones_like(valid))

    valid &= torch.where(labels == light, light_rule, torch.ones_like(valid))
    valid &= torch.where(labels == sign, sign_rule, torch.ones_like(valid))
    valid &= torch.where(labels == truck, truck_rule, torch.ones_like(valid))
    valid &= torch.where(labels == bus, bus_rule, torch.ones_like(valid))

    boxes = boxes[valid]
    scores = scores[valid]
    labels = labels[valid]

    if scores.numel() == 0:
        return boxes, scores, labels

    # 같은 클래스 중복 제거
    offset = labels.float()[:, None] * (img_size + 1)
    keep = nms(boxes + offset, scores, 0.50)
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    if scores.numel() == 0:
        return boxes, scores, labels

    # car / truck / bus가 거의 같은 위치에 겹치면 가장 높은 점수 하나만 남긴다.
    keep = suppress_cross_class_duplicates(
        boxes,
        scores,
        labels,
        group_classes=[car, truck, bus],
        iou_th=0.70
    )
    boxes = boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    # 한 이미지에 같은 클래스가 과하게 뜨는 현상 완화
    class_limits = {
        truck: 3,
        bus: 2,
        light: 12,
        sign: 12,
        pedestrian: 20,
        rider: 8,
        motorcycle: 8,
        bicycle: 8,
    }

    boxes, scores, labels = limit_predictions_per_class(
        boxes,
        scores,
        labels,
        limits=class_limits
    )

    return boxes, scores, labels


@torch.no_grad()
def detect(
    model,
    images,
    score_th=0.05,
    iou_th=0.50,
    pre_nms_topk=1200,
    max_det=60,
    use_rules=False
):
    model.eval()
    images = images.to(device, non_blocking=True)
    outputs = model(images)

    cls = outputs["cls"]
    reg = outputs["reg"]
    ctr = outputs["ctr"]
    stride = outputs["stride"]
    feat_h, feat_w = outputs["feat_hw"]

    centers = make_grid(feat_h, feat_w, stride, cls.device)

    cls_prob = torch.sigmoid(cls)
    ctr_prob = torch.sigmoid(ctr)

    results = []

    for i in range(images.shape[0]):
        cls_score, labels = cls_prob[i].max(dim=1)
        scores = cls_score * ctr_prob[i]

        keep_pos = torch.where(scores > score_th)[0]
        labels = labels[keep_pos]
        scores = scores[keep_pos]

        extra_pos = []
        extra_cls = []
        extra_scores = []

        # 작은 객체만 낮은 threshold로 추가 후보를 살린다.
        extra_class_thresholds = {
            class_to_idx["traffic light"]: 0.030,
            class_to_idx["traffic sign"]: 0.035,
        }

        for c, th in extra_class_thresholds.items():
            s = cls_prob[i, :, c] * ctr_prob[i]
            k = torch.where(s > th)[0]

            if k.numel() > 0:
                extra_pos.append(k)
                extra_cls.append(torch.full_like(k, c))
                extra_scores.append(s[k])

        if len(extra_pos) > 0:
            keep_pos = torch.cat([keep_pos] + extra_pos)
            labels = torch.cat([labels] + extra_cls)
            scores = torch.cat([scores] + extra_scores)

        if scores.numel() == 0:
            results.append(_empty_pred())
            continue

        if scores.numel() > pre_nms_topk:
            top_idx = torch.argsort(scores, descending=True)[:pre_nms_topk]
            keep_pos = keep_pos[top_idx]
            labels = labels[top_idx]
            scores = scores[top_idx]

        boxes = ltrb_to_xyxy(centers[keep_pos], reg[i, keep_pos])
        boxes[:, [0, 2]] = boxes[:, [0, 2]].clamp(0, IMG_SIZE)
        boxes[:, [1, 3]] = boxes[:, [1, 3]].clamp(0, IMG_SIZE)

        bw = boxes[:, 2] - boxes[:, 0]
        bh = boxes[:, 3] - boxes[:, 1]
        area = bw * bh
        aspect = bw / bh.clamp(min=1.0)

        valid = (
            (bw > 2) &
            (bh > 2) &
            (area > 4) &
            (area < IMG_SIZE * IMG_SIZE * 0.80) &
            (aspect > 0.05) &
            (aspect < 16.0)
        )

        boxes = boxes[valid]
        scores = scores[valid]
        labels = labels[valid]

        if scores.numel() == 0:
            results.append(_empty_pred())
            continue

        offset = labels.float()[:, None] * (IMG_SIZE + 1)
        keep_idx = nms(boxes + offset, scores, iou_th)

        boxes = boxes[keep_idx]
        scores = scores[keep_idx]
        labels = labels[keep_idx]

        if use_rules:
            boxes, scores, labels = apply_detection_rules(boxes, scores, labels)

        if scores.numel() > max_det:
            top_idx = torch.argsort(scores, descending=True)[:max_det]
            boxes = boxes[top_idx]
            scores = scores[top_idx]
            labels = labels[top_idx]

        results.append({
            "boxes": boxes.detach().cpu(),
            "scores": scores.detach().cpu(),
            "labels": labels.detach().cpu(),
        })

    return results

## 16. mAP@0.5 평가 함수

mAP@0.5는 예측 box와 정답 box의 IoU가 0.5 이상이면 맞춘 것으로 보고 계산한다.  
클래스별 AP를 구한 뒤 평균을 내면 mAP가 된다.

In [ ]:
def compute_ap(recall, precision):
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))

    for i in range(mpre.size - 1, 0, -1):
        mpre[i - 1] = np.maximum(mpre[i - 1], mpre[i])

    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1])


@torch.no_grad()
def evaluate_map50(model, loader, score_th=0.05, iou_th=0.5, max_batches=None, use_rules=True):
    model.eval()
    all_preds = []
    all_gts = []

    for batch_idx, (images, targets) in enumerate(tqdm(loader, desc="evaluate")):
        preds = detect(
            model,
            images,
            score_th=score_th,
            iou_th=iou_th,
            use_rules=use_rules
        )

        for pred, target in zip(preds, targets):
            all_preds.append(pred)
            all_gts.append({
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu(),
            })

        if max_batches is not None and batch_idx + 1 >= max_batches:
            break

    aps = []
    ap_by_class = {}

    for c in range(NUM_CLASSES):
        preds_c = []
        npos = 0

        for gt in all_gts:
            gt_mask = gt["labels"] == c
            npos += int(gt_mask.sum().item())

        if npos == 0:
            ap_by_class[idx_to_class[c]] = np.nan
            continue

        for img_id, pred in enumerate(all_preds):
            pred_mask = pred["labels"] == c
            boxes = pred["boxes"][pred_mask]
            scores = pred["scores"][pred_mask]

            for b, s in zip(boxes, scores):
                preds_c.append((img_id, float(s.item()), b))

        preds_c.sort(key=lambda x: x[1], reverse=True)

        tp = np.zeros(len(preds_c))
        fp = np.zeros(len(preds_c))
        matched = defaultdict(set)

        for i, (img_id, score, pbox) in enumerate(preds_c):
            gt = all_gts[img_id]
            gt_mask = gt["labels"] == c
            gt_boxes = gt["boxes"][gt_mask]

            if gt_boxes.numel() == 0:
                fp[i] = 1
                continue

            ious = box_iou(pbox.unsqueeze(0), gt_boxes).squeeze(0)
            max_iou, max_idx = ious.max(dim=0)
            max_idx = int(max_idx.item())

            if max_iou.item() >= iou_th and max_idx not in matched[img_id]:
                tp[i] = 1
                matched[img_id].add(max_idx)
            else:
                fp[i] = 1

        if len(preds_c) == 0:
            ap = 0.0
        else:
            tp_cum = np.cumsum(tp)
            fp_cum = np.cumsum(fp)
            recall = tp_cum / max(npos, 1)
            precision = tp_cum / np.maximum(tp_cum + fp_cum, 1e-6)
            ap = compute_ap(recall, precision)

        aps.append(ap)
        ap_by_class[idx_to_class[c]] = ap

    map50 = float(np.nanmean(aps)) if len(aps) > 0 else 0.0
    return map50, ap_by_class

## 17. confidence threshold 선택

같은 모델이라도 threshold에 따라 mAP가 달라진다.  
일부 시험 데이터에서 먼저 빠르게 비교하고, 가장 좋은 값을 전체 평가에 사용했다.

In [ ]:
def threshold_sweep(
    model,
    loader,
    name,
    thresholds=None,
    max_batches=50,
    use_rules=True
):
    if thresholds is None:
        thresholds = [0.05, 0.07, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]

    important_classes = ["car", "truck", "bus", "traffic light", "traffic sign"]

    rows = []

    for th in thresholds:
        score, ap_by_class = evaluate_map50(
            model,
            loader,
            score_th=th,
            iou_th=0.5,
            max_batches=max_batches,
            use_rules=use_rules
        )

        row = {
            "model": name,
            "use_rules": use_rules,
            "score_th": th,
            "preview_mAP@0.5": score,
            "preview_percent": score * 100,
        }

        for cls_name in important_classes:
            row[f"{cls_name}_AP50"] = ap_by_class.get(cls_name, np.nan)

        rows.append(row)

    df = pd.DataFrame(rows).sort_values("preview_mAP@0.5", ascending=False)
    display(df)

    best_th = float(df.iloc[0]["score_th"])
    print(f"{name} best threshold:", best_th)

    return best_th, df


lecture_best_th, lecture_th_df = threshold_sweep(
    lecture_model,
    test_loader,
    "lecture_raw",
    max_batches=50,
    use_rules=False
)

improved_best_th, improved_th_df = threshold_sweep(
    improved_model,
    test_loader,
    "improved_rules",
    max_batches=50,
    use_rules=True
)

improved_raw_best_th, improved_raw_th_df = threshold_sweep(
    improved_model,
    test_loader,
    "improved_raw",
    max_batches=50,
    use_rules=False
)

## 18. 전체 시험 데이터 평가

앞에서 찾은 threshold를 사용해 시험 데이터 3000장 전체에서 mAP@0.5를 계산한다.

In [ ]:
lecture_map50, lecture_ap = evaluate_map50(
    lecture_model,
    test_loader,
    score_th=lecture_best_th,
    iou_th=0.5,
    use_rules=False
)

improved_map50, improved_ap = evaluate_map50(
    improved_model,
    test_loader,
    score_th=improved_raw_best_th,
    iou_th=0.5,
    use_rules=False
)

summary_df = pd.DataFrame({
    "model": ["lecture_model", "improved_model"],
    "threshold": [lecture_best_th, improved_raw_best_th],
    "mAP@0.5": [lecture_map50, improved_map50],
    "mAP@0.5(%)": [lecture_map50 * 100, improved_map50 * 100],
    "params": [
        count_parameters(lecture_model),
        count_parameters(improved_model),
    ],
})

result_df = pd.DataFrame({
    "class": CLASSES,
    "lecture_AP50": [lecture_ap.get(c, np.nan) for c in CLASSES],
    "improved_AP50": [improved_ap.get(c, np.nan) for c in CLASSES],
})

result_df["change"] = result_df["improved_AP50"] - result_df["lecture_AP50"]

display(summary_df)
display(result_df)

In [ ]:
metric_explain_df = pd.DataFrame({
    "metric": ["IoU", "Precision", "Recall", "AP@0.5", "mAP@0.5"],
    "meaning": [
        "예측 box와 정답 box가 얼마나 겹치는지 보는 값",
        "모델이 맞다고 예측한 것 중 진짜 맞은 비율",
        "실제 객체 중 모델이 찾아낸 비율",
        "한 클래스에서 confidence threshold 변화에 따른 Precision-Recall 성능",
        "각 클래스 AP@0.5의 평균"
    ],
    "low_when": [
        "box 위치나 크기가 정답과 많이 다를 때",
        "허공 box, 잘못된 클래스 예측이 많을 때",
        "실제 객체를 많이 놓칠 때",
        "그 클래스의 검출이 전반적으로 불안정할 때",
        "여러 클래스에서 검출 성능이 낮을 때"
    ]
})

display(metric_explain_df)

## 19. 결과 해석

기준 모델은 23강의 FCOS 구조를 BDD100K에 맞춰 적용한 모델이다.  
개선 모델은 같은 FCOS 흐름을 유지하면서, BDD100K에서 특히 문제가 되는 작은 객체 탐지에 더 맞게 수정했다.

전체 결과를 보면 개선 모델의 mAP@0.5가 기준 모델보다 높게 나왔다.  
기준 모델은 약 10.25%, 개선 모델은 약 14.51%로, 전체적으로는 개선 모델이 더 좋은 성능을 보였다.

개선 효과가 가장 크게 나타난 부분은 `traffic light`와 `traffic sign`이다.  
이 두 클래스는 이미지 안에서 크기가 작고 멀리 있는 경우가 많아서 놓치기 쉬운데, 입력 크기를 512로 사용하고 feature map stride를 8로 줄인 것이 도움이 되었다고 볼 수 있다.


1. 입력 크기를 512로 사용해 작은 객체가 덜 뭉개지게 했다.
2. 최종 feature map을 stride 8로 만들어 더 촘촘한 격자에서 예측했다.
3. backbone은 ResNet-18 layer3보다 작은 커스텀 구조로 만들었다.
4. focal loss를 사용해 배경 칸이 많은 객체 탐지의 불균형을 줄였다.
5. class weight를 약하게 적용해 적게 나오는 클래스가 완전히 무시되지 않게 했다.
6. threshold를 고정하지 않고 여러 값으로 비교해 가장 나은 값을 사용했다.

다만 모든 클래스가 좋아진 것은 아니다.  
`truck`은 기준 모델보다 AP가 낮게 나왔다. 실제 시각화에서도 truck이 car나 bus와 헷갈리는 경우가 있었다.  
BDD100K에서는 멀리 있는 차량이나 큰 차량이 비슷하게 보이는 경우가 많아서, 단순히 작은 객체를 더 잘 보게 만드는 것만으로는 truck/car/bus 구분까지 완전히 해결되지는 않았다.

후처리 규칙도 따로 실험했다.  
truck이나 bus의 허공 예측을 줄이기 위해 box 크기, 위치, 클래스별 개수 제한을 적용해 보았지만, 전체적으로는 후처리 전후 차이가 크지 않았다.  
그래서 최종 평가는 인위적인 후처리보다 모델의 raw prediction을 기준으로 진행했다.

결론적으로 개선 모델은 작은 객체 탐지 성능을 높이는 데 효과가 있었고, 특히 traffic light와 traffic sign에서 뚜렷한 향상이 있었다.  
반면 truck처럼 car, bus와 모양이 비슷한 클래스는 아직 혼동이 남아 있어 향후 개선이 필요한 부분이다.

## 20. 시험 이미지 10장 시각화

정답 라벨, 강의자료 모델 예측, 개선 모델 예측을 같은 이미지에서 비교한다.  
mAP 수치만 보면 전체 성능은 알 수 있지만, 실제 box가 어디에 찍히는지는 직접 확인해야 한다.

시각화에서 중점적으로 본 부분은 세 가지이다.

1. 정답 box 근처에 예측 box가 잘 생기는지
2. traffic light와 traffic sign처럼 작은 객체가 이전보다 잘 잡히는지
3. car, truck, bus가 서로 헷갈리는 경우가 얼마나 남아있는지

결과를 보면 개선 모델은 작은 신호등과 표지판을 기준 모델보다 더 자주 잡는 경향이 있었다.  
다만 threshold를 너무 낮게 두면 허공에 box가 많이 생기기 때문에, 시각화에서는 평가용 threshold와 별도로 보기 쉬운 threshold도 함께 확인했다.

특히 truck은 여전히 car나 bus와 혼동되는 경우가 있었다.  

In [ ]:
def compare_prediction_to_answer(pred, target, iou_tp=0.5, iou_near=0.1):
    pred_boxes = pred["boxes"].cpu()
    pred_labels = pred["labels"].cpu()
    pred_scores = pred["scores"].cpu()

    gt_boxes = target["boxes"].cpu()
    gt_labels = target["labels"].cpu()

    rows = []
    matched_gt = set()

    if pred_boxes.numel() > 0 and gt_boxes.numel() > 0:
        iou_mat = box_iou(pred_boxes, gt_boxes)
    else:
        iou_mat = torch.zeros((len(pred_boxes), len(gt_boxes)))

    order = torch.argsort(pred_scores, descending=True) if len(pred_scores) > 0 else []

    for pi_tensor in order:
        pi = int(pi_tensor)
        p_label = int(pred_labels[pi])
        p_name = idx_to_class[p_label]
        score = float(pred_scores[pi])

        if len(gt_boxes) == 0:
            rows.append({
                "status": "허공 FP",
                "pred": p_name,
                "gt": "",
                "score": round(score, 3),
                "iou": 0.0,
            })
            continue

        ious = iou_mat[pi]
        max_iou, max_idx = ious.max(dim=0)
        max_iou = float(max_iou)
        max_idx = int(max_idx)
        gt_label = int(gt_labels[max_idx])
        gt_name = idx_to_class[gt_label]

        if max_iou < iou_near:
            rows.append({
                "status": "허공 FP",
                "pred": p_name,
                "gt": "",
                "score": round(score, 3),
                "iou": round(max_iou, 3),
            })
            continue

        same_class = p_label == gt_label

        if max_iou >= iou_tp:
            if same_class and max_idx not in matched_gt:
                status = "정답 TP"
                matched_gt.add(max_idx)
            elif same_class and max_idx in matched_gt:
                status = "중복 예측"
            else:
                status = "클래스 혼동"
        else:
            if same_class:
                status = "box 불안정"
            else:
                status = "근처 클래스 혼동"

        rows.append({
            "status": status,
            "pred": p_name,
            "gt": gt_name,
            "score": round(score, 3),
            "iou": round(max_iou, 3),
        })

    for gi in range(len(gt_boxes)):
        if gi not in matched_gt:
            rows.append({
                "status": "미검출 FN",
                "pred": "",
                "gt": idx_to_class[int(gt_labels[gi])],
                "score": np.nan,
                "iou": np.nan,
            })

    return pd.DataFrame(rows)


def summarize_compare(df):
    order = [
        "정답 TP",
        "box 불안정",
        "중복 예측",
        "클래스 혼동",
        "근처 클래스 혼동",
        "허공 FP",
        "미검출 FN",
    ]

    counts = df["status"].value_counts().reindex(order).fillna(0).astype(int)

    return pd.DataFrame({
        "status": counts.index,
        "count": counts.values,
    })


@torch.no_grad()
def visualize_before_after_rules(indices=None, n=8, score_th=None, max_det=30):
    if indices is None:
        indices = random.sample(range(len(test_ds)), n)

    if score_th is None:
        score_th = improved_best_th

    for idx in indices:
        img, target = test_ds[idx]
        img_np = denormalize_image(img)

        raw_pred = detect(
            improved_model,
            img.unsqueeze(0),
            score_th=score_th,
            max_det=max_det,
            use_rules=False
        )[0]

        fixed_pred = detect(
            improved_model,
            img.unsqueeze(0),
            score_th=score_th,
            max_det=max_det,
            use_rules=True
        )[0]

        raw_df = compare_prediction_to_answer(raw_pred, target)
        fixed_df = compare_prediction_to_answer(fixed_pred, target)

        print("image:", target["image_id"])

        print("후처리 전 요약")
        display(summarize_compare(raw_df))

        print("후처리 후 요약")
        display(summarize_compare(fixed_df))

        print("후처리 전 문제 목록")
        display(raw_df.query("status != '정답 TP'").head(20))

        print("후처리 후 문제 목록")
        display(fixed_df.query("status != '정답 TP'").head(20))

        gt_img = draw_boxes_on_image(
            img_np,
            target["boxes"].numpy(),
            target["labels"].numpy(),
            scores=None,
            width=2
        )

        raw_img = draw_boxes_on_image(
            img_np,
            raw_pred["boxes"].numpy(),
            raw_pred["labels"].numpy(),
            raw_pred["scores"].numpy(),
            width=2
        )

        fixed_img = draw_boxes_on_image(
            img_np,
            fixed_pred["boxes"].numpy(),
            fixed_pred["labels"].numpy(),
            fixed_pred["scores"].numpy(),
            width=2
        )

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(gt_img)
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(raw_img)
        axes[1].set_title("Before Rules")
        axes[1].axis("off")

        axes[2].imshow(fixed_img)
        axes[2].set_title("After Rules")
        axes[2].axis("off")

        plt.suptitle(target["image_id"])
        plt.tight_layout()
        plt.show()

    return indices


In [ ]:
check_indices = visualize_before_after_rules(
    indices=None,
    n=10,
    score_th=0.1,
    max_det=15
)

check_indices

In [ ]:
@torch.no_grad()
def get_single_prediction(
    model,
    image_tensor,
    score_th=0.10,
    iou_th=0.5,
    max_det=40,
    use_rules=True
):
    model.eval()

    preds = detect(
        model,
        image_tensor.unsqueeze(0),
        score_th=score_th,
        iou_th=iou_th,
        max_det=max_det,
        use_rules=use_rules
    )[0]

    return preds


def visualize_comparison(
    indices=None,
    n=10,
    max_det=40,
    lecture_th=None,
    improved_th=None,
    improved_use_rules=True
):
    if indices is None:
        indices = random.sample(range(len(test_ds)), n)

    if lecture_th is None:
        lecture_th = lecture_best_th

    if improved_th is None:
        improved_th = improved_best_th

    for idx in indices:
        img, target = test_ds[idx]
        img_np = denormalize_image(img)

        gt_img = draw_boxes_on_image(
            img_np,
            target["boxes"].numpy(),
            target["labels"].numpy(),
            scores=None,
            width=2
        )

        lecture_pred = get_single_prediction(
            lecture_model,
            img,
            score_th=lecture_th,
            max_det=max_det,
            use_rules=False
        )

        improved_pred = get_single_prediction(
            improved_model,
            img,
            score_th=improved_th,
            max_det=max_det,
            use_rules=improved_use_rules
        )

        lecture_img = draw_boxes_on_image(
            img_np,
            lecture_pred["boxes"].numpy(),
            lecture_pred["labels"].numpy(),
            lecture_pred["scores"].numpy(),
            width=2
        )

        improved_img = draw_boxes_on_image(
            img_np,
            improved_pred["boxes"].numpy(),
            improved_pred["labels"].numpy(),
            improved_pred["scores"].numpy(),
            width=2
        )

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))

        axes[0].imshow(gt_img)
        axes[0].set_title("Ground Truth")
        axes[0].axis("off")

        axes[1].imshow(lecture_img)
        axes[1].set_title(f"Lecture Raw / th={lecture_th}")
        axes[1].axis("off")

        rule_text = "Rules" if improved_use_rules else "Raw"
        axes[2].imshow(improved_img)
        axes[2].set_title(f"Improved {rule_text} / th={improved_th}")
        axes[2].axis("off")

        plt.suptitle(target["image_id"])
        plt.tight_layout()
        plt.show()

    return indices


# 평가 기준과 맞춰 보는 용도
vis_indices_eval = visualize_comparison(
    indices=None,
    n=10,
    max_det=10,
    lecture_th=lecture_best_th,
    improved_th=improved_best_th,
    improved_use_rules=True
)

vis_indices_eval

## 21. 최종 정리

이번 실험에서는 BDD100K val 이미지 10000장을 파일 이름 순서대로 나누어 사용했다.  
앞 7000장은 학습 데이터, 뒤 3000장은 시험 데이터로 사용했다.

모델은 23강의 FCOS 구조를 기준으로 잡았다.  
FCOS는 각 격자점에서 box의 네 방향 거리를 예측하는 방식이다. anchor를 직접 정하지 않아도 되어 구조가 비교적 단순하고, class head, box head, centerness head로 역할이 나뉜다.

개선 모델은 작은 객체를 더 잘 잡는 방향으로 수정했다.  
입력 크기를 512로 키우고, 최종 feature map의 stride를 8로 줄여 이미지를 더 촘촘한 격자로 보게 했다.  
BDD100K에는 traffic light와 traffic sign처럼 작고 멀리 있는 객체가 많기 때문에 이 부분이 가장 중요한 개선 방향이었다.

또한 focal loss와 class weight를 사용했다.  
객체 탐지에서는 대부분의 격자칸이 배경이라서, 그냥 학습하면 쉬운 배경 칸의 영향이 너무 커질 수 있다.  
focal loss는 이런 쉬운 배경보다 헷갈리는 객체 후보에 더 집중하게 해준다. class weight는 적게 등장하는 클래스가 완전히 무시되지 않도록 약하게 보정하는 역할을 한다.

최종 결과에서 개선 모델은 기준 모델보다 mAP@0.5가 높게 나왔다.  
기준 모델은 약 10.25%, 개선 모델은 약 14.51%로 전체 성능이 향상되었다.  
특히 traffic light와 traffic sign의 AP가 크게 올랐기 때문에, 작은 객체를 더 잘 보게 만든 개선 방향은 효과가 있었다고 볼 수 있다.

하지만 truck 클래스는 오히려 낮아졌다.  
truck은 car, bus와 모양이 비슷하고, 멀리 있으면 크기도 작아져서 모델이 헷갈리기 쉽다.  
실제 시각화에서도 truck을 car로 보거나, 반대로 애매한 차량을 truck처럼 예측하는 경우가 있었다.

후처리 규칙도 실험했지만 최종 결과에는 기본 적용하지 않았다.  
box 크기나 위치 조건으로 일부 이상한 예측을 줄일 수는 있었지만, 전체적으로는 후처리 전후 차이가 크지 않았고 오히려 맞는 box를 지울 위험도 있었다.  
그래서 최종 평가는 개선 모델의 raw prediction을 기준으로 정리했다.

따라서 이번 개선의 핵심은 작은 객체 탐지 성능 향상이다.  
traffic light와 traffic sign은 좋아졌고, 전체 mAP도 올랐다.  
반면 truck/car/bus처럼 비슷한 차량 클래스의 구분은 아직 한계로 남았다.

## 요약

이 과제에서는 자동차 블랙박스 같은 도로 이미지에서 사람, 자동차, 버스, 신호등, 표지판 같은 객체를 찾았다.  
단순히 이미지에 무엇이 있는지만 맞추는 것이 아니라, 그 객체가 어디에 있는지 box까지 맞춰야 한다.

기준 모델은 23강에서 다룬 FCOS 방식이다.  
FCOS는 이미지를 격자로 나누고, 각 격자점이 객체의 중심 근처인지 판단하면서 box의 왼쪽, 위쪽, 오른쪽, 아래쪽 거리를 예측한다.

개선 모델에서는 작은 객체를 더 잘 보기 위해 이미지를 512 크기로 사용하고, 더 촘촘한 격자에서 예측하도록 만들었다.  
BDD100K에는 신호등이나 표지판처럼 작은 물체가 많기 때문에 이 부분이 중요하다.

최종 결과에서 개선 모델은 기준 모델보다 mAP@0.5가 높았다.  
특히 traffic light와 traffic sign의 성능 향상이 컸다.  
그래서 이번 개선은 작은 객체를 더 잘 잡는 방향에서는 효과가 있었다.

다만 truck은 기준 모델보다 낮게 나왔다.  
truck은 car나 bus와 비슷하게 보이는 경우가 많아서, 작은 객체를 잘 보게 만드는 개선만으로는 완전히 해결되지 않았다.

결론적으로 이번 실험의 가장 중요한 점은 다음과 같다.  
입력 크기와 stride 조정은 작은 객체 탐지에 도움이 되었고, 전체 성능도 향상되었다.  
하지만 차량 종류를 더 정확히 구분하기 위해서는 truck, bus, car를 더 잘 분리할 수 있는 추가 개선이 필요하다.